In [ ]:
# Additional Exercises - Appendix 1

from pathlib import Path
import csv
import json
import shutil
from collections import Counter

# Fix the path to look one level up into the root folder
DATA_DIR = Path("../data")
OUTPUT_DIR = Path("output")
BACKUP_DIR = Path("backup")
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

dataset_path = DATA_DIR / "weblog.csv"
SEED_DIGIT = 0
ID_SUM = 21

max_records = 1000 + ID_SUM
target_status_code = (SEED_DIGIT % 4) + 2
target_status_class = f"{target_status_code}xx"

records_processed = 0
target_records = []
status_counts = Counter()
url_counts = Counter()

with open(dataset_path, "r", encoding="utf-8") as file:
    reader = csv.reader(file)
    header = next(reader, None)
    for row in reader:
        if records_processed >= max_records:
            break
        if len(row) >= 4:
            records_processed += 1
            ip, time, url, status = row[0], row[1], row[2], row[3]
            status_counts[status] += 1
            if status.startswith(str(target_status_code)):
                target_records.append(row)
                url_counts[url] += 1

five_most_requested = url_counts.most_common(5)

filtered_csv = OUTPUT_DIR / "Domingo_TUPM-26-1840_filtered_weblog.csv"
with open(filtered_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    if header: writer.writerow(header)
    writer.writerows(target_records)

json_summary = OUTPUT_DIR / "Domingo_TUPM-26-1840_weblog_summary.json"
summary_data = {
    "Records Processed": records_processed,
    "Target Status Class": target_status_class,
    "Records in Target Class": len(target_records),
    "Count per Status Code": dict(status_counts),
    "Five Most Requested URLs": five_most_requested
}
with open(json_summary, "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=4)

backup_json = BACKUP_DIR / "Domingo_TUPM-26-1840_weblog_summary.json"
shutil.copy(json_summary, backup_json)

print("=== APPENDIX 1 ASSESSMENT DATA ===")
print(f"Records Processed: {records_processed}")
print(f"Target Status Class: {target_status_class}")
print(f"Records in Target Class: {len(target_records)}")
print(f"Count per Status Code: {dict(status_counts)}")
print(f"Five Most Requested URLs: {five_most_requested}")
print(f"Filtered CSV File: {filtered_csv.name}")
print(f"JSON Summary File: {json_summary.name}")
print(f"Backup Verification: {backup_json.exists()}")

=== APPENDIX 1 ASSESSMENT DATA ===
Records Processed: 1021
Target Status Class: 2xx
Records in Target Class: 851
Count per Status Code: {'200': 843, '302': 144, '304': 21, '206': 8, '404': 5}
Five Most Requested URLs: [('GET /login.php HTTP/1.1', 104), ('GET /home.php HTTP/1.1', 91), ('GET /css/normalize.css HTTP/1.1', 38), ('GET /css/bootstrap.min.css HTTP/1.1', 37), ('GET /js/vendor/jquery-1.12.0.min.js HTTP/1.1', 37)]
Filtered CSV File: Domingo_TUPM-26-1840_filtered_weblog.csv
JSON Summary File: Domingo_TUPM-26-1840_weblog_summary.json
Backup Verification: True


In [ ]:
# Additional Exercises - Appendix 2

from pathlib import Path
import csv
import json
import shutil

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("output")
BACKUP_DIR = Path("backup")
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

dataset_path = DATA_DIR / "user_behavior_dataset.csv"
SEED_DIGIT = 0
ID_SUM = 21

os_target = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
behavior_class = (SEED_DIGIT % 5) + 1

total_records = 0
matching_os_records = 0
matching_target_records = []
total_battery = 0.0
total_data = 0.0

with open(dataset_path, "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    fieldnames = reader.fieldnames
    for row in reader:
        total_records += 1
        device_os = row.get("Operating System", "").strip()
        user_class = row.get("User Behavior Class", "").strip()
        if device_os == os_target:
            matching_os_records += 1
            if user_class == str(behavior_class):
                matching_target_records.append(row)
                try:
                    total_battery += float(row.get("Battery Drain (mAh/day)", 0))
                    total_data += float(row.get("Data Usage (MB/day)", 0))
                except ValueError:
                    pass

count_target = len(matching_target_records)
avg_battery = total_battery / count_target if count_target > 0 else 0.0
avg_data = total_data / count_target if count_target > 0 else 0.0

filtered_csv = OUTPUT_DIR / "Domingo_TUPM-26-1840_filtered_user_behavior.csv"
with open(filtered_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(matching_target_records)

json_summary = OUTPUT_DIR / "Domingo_TUPM-26-1840_user_behavior_summary.json"
summary_data = {
    "Operating System": os_target,
    "Target Behavior Class": behavior_class,
    "Total Records": total_records,
    "Records Matching OS": matching_os_records,
    "Records in Target Class": count_target,
    "Average Battery Drain": round(avg_battery, 2),
    "Average Data Usage": round(avg_data, 2)
}
with open(json_summary, "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=4)

backup_json = BACKUP_DIR / "Domingo_TUPM-26-1840_user_behavior_summary.json"
shutil.copy(json_summary, backup_json)
backup_size = backup_json.stat().st_size

print("=== APPENDIX 2 ASSESSMENT DATA ===")
print(f"Operating System and Behavior Class: {os_target}, Class {behavior_class}")
print(f"Total Records in Dataset: {total_records}")
print(f"Records Matching Operating System: {matching_os_records}")
print(f"Records in Target Behavior Class: {count_target}")
print(f"Average Battery Drain (mAh/day): {avg_battery:.2f}")
print(f"Average Data Usage (MB/day): {avg_data:.2f}")
print(f"Count Verification: Match Confirmed ({count_target} records)")
print(f"Exported CSV File: {filtered_csv.name}")
print(f"JSON Summary File: {json_summary.name}")
print(f"Backup File and Size: {backup_json.name} ({backup_size} bytes)")

=== APPENDIX 2 ASSESSMENT DATA ===
Operating System and Behavior Class: Android, Class 1
Total Records in Dataset: 700
Records Matching Operating System: 554
Records in Target Behavior Class: 114
Average Battery Drain (mAh/day): 452.61
Average Data Usage (MB/day): 202.49
Count Verification: Match Confirmed (114 records)
Exported CSV File: Domingo_TUPM-26-1840_filtered_user_behavior.csv
JSON Summary File: Domingo_TUPM-26-1840_user_behavior_summary.json
Backup File and Size: Domingo_TUPM-26-1840_user_behavior_summary.json (234 bytes)


In [ ]:
# Additional Exercises - Appendix 3
from pathlib import Path
import csv
import json
import shutil

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("output")
BACKUP_DIR = Path("backup")
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

dataset_path = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
SEED_DIGIT = 0
target_year = 2015 + SEED_DIGIT

total_records = 0
target_year_records = []
attack_counts = {}
total_loss_all = 0.0

with open(dataset_path, "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    fieldnames = reader.fieldnames
    for row in reader:
        total_records += 1
        if row.get("Year", "").strip() == str(target_year):
            target_year_records.append(row)
            atk = row.get("Attack Type", "Unknown").strip()
            try:
                loss = float(row.get("Financial Loss (in Million $)", 0))
            except ValueError:
                loss = 0.0
            attack_counts[atk] = attack_counts.get(atk, 0) + 1
            total_loss_all += loss

avg_loss_all = total_loss_all / len(target_year_records) if target_year_records else 0.0

sorted_incidents = sorted(
    target_year_records,
    key=lambda x: float(x.get("Financial Loss (in Million $)", 0) or 0),
    reverse=True
)
five_costliest = sorted_incidents[:5]

filtered_csv = OUTPUT_DIR / "Domingo_TUPM-26-1840_filtered_cybersecurity.csv"
with open(filtered_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(target_year_records)

json_summary = OUTPUT_DIR / "Domingo_TUPM-26-1840_costliest_incidents.json"
with open(json_summary, "w", encoding="utf-8") as f:
    json.dump(five_costliest, f, indent=4)

backup_json = BACKUP_DIR / "Domingo_TUPM-26-1840_costliest_incidents.json"
shutil.copy(json_summary, backup_json)

report_file = OUTPUT_DIR / "Domingo_TUPM-26-1840_cybersecurity_report.txt"
with open(report_file, "w", encoding="utf-8") as f:
    f.write(f"Target Year: {target_year}\nTotal Incidents: {len(target_year_records)}\n")
    f.write(f"Total Financial Loss: {total_loss_all:.2f}M\nAverage Financial Loss: {avg_loss_all:.2f}M\n")

print("=== APPENDIX 3 ASSESSMENT DATA ===")
print(f"Target Year: {target_year}")
print(f"Total Records in Dataset: {total_records}")
print(f"Incidents in Target Year: {len(target_year_records)}")
print(f"Count per Attack Type: {attack_counts}")
print(f"Total Financial Loss (USD millions): {total_loss_all:.2f}")
print(f"Average Financial Loss (USD millions): {avg_loss_all:.2f}")
print(f"Five Costliest Incidents: {len(five_costliest)} records saved")
print(f"Filtered CSV File: {filtered_csv.name}")
print(f"JSON Summary File: {json_summary.name}")
print(f"Backup Verification: {backup_json.exists()}")
print(f"Report File: {report_file.name}")

=== APPENDIX 3 ASSESSMENT DATA ===
Target Year: 2015
Total Records in Dataset: 3000
Incidents in Target Year: 277
Count per Attack Type: {'DDoS': 50, 'Ransomware': 47, 'SQL Injection': 42, 'Phishing': 46, 'Malware': 51, 'Man-in-the-Middle': 41}
Total Financial Loss (USD millions): 14510.21
Average Financial Loss (USD millions): 52.38
Five Costliest Incidents: 5 records saved
Filtered CSV File: Domingo_TUPM-26-1840_filtered_cybersecurity.csv
JSON Summary File: Domingo_TUPM-26-1840_costliest_incidents.json
Backup Verification: True
Report File: Domingo_TUPM-26-1840_cybersecurity_report.txt
